In [2]:
# Hotel Pricing Lakehouse Project - Gold Modeling

import pandas as pd
import numpy as np
from pathlib import Path

SILVER_PATH = Path("C:\\Users\\cxpen\\Documents\\JOB\\Portfolios\\hotel-pricing-lakehouse\\data\\silver\\hotel_bookings_silver.csv")
GOLD_DIR = Path("C:\\Users\\cxpen\\Documents\\JOB\\Portfolios\\hotel-pricing-lakehouse\\data\\gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SILVER_PATH)

print(df.shape)
df.head()

(86638, 42)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,ingestion_timestamp,source_file_name,batch_id,total_guests,arrival_date,total_nights,booking_status,estimated_revenue,silver_processed_timestamp,meal_plan
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,1.0,2015-07-01,1,Not Canceled,75.0,2026-05-16 16:13:00.384678,Bed and Breakfast
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,1.0,2015-07-01,1,Not Canceled,75.0,2026-05-16 16:13:00.384678,Bed and Breakfast
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,196.0,2026-05-16 16:13:00.384678,Bed and Breakfast
3,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,214.0,2026-05-16 16:13:00.384678,Bed and Breakfast
4,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,2026-05-16 10:44:39.377975,hotel_bookings.csv,4f7e4a36-3ea5-4840-9c9c-9422ea391bd5,2.0,2015-07-01,2,Not Canceled,206.0,2026-05-16 16:13:00.384678,Full Board


In [5]:
dim_hotel = (
    df[["hotel"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_hotel["hotel_id"] = dim_hotel.index + 1

dim_hotel = dim_hotel[["hotel_id", "hotel"]]

dim_hotel

,hotel_id,hotel
0,1,Resort Hotel
1,2,City Hotel


In [6]:
df["arrival_date"] = pd.to_datetime(df["arrival_date"])

dim_date = (
    df[["arrival_date"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_date["date_id"] = dim_date["arrival_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["arrival_date"].dt.year
dim_date["month"] = dim_date["arrival_date"].dt.month
dim_date["month_name"] = dim_date["arrival_date"].dt.month_name()
dim_date["day"] = dim_date["arrival_date"].dt.day
dim_date["quarter"] = dim_date["arrival_date"].dt.quarter
dim_date["is_weekend"] = dim_date["arrival_date"].dt.dayofweek >= 5

dim_date = dim_date[
    [
        "date_id",
        "arrival_date",
        "year",
        "month",
        "month_name",
        "day",
        "quarter",
        "is_weekend"
    ]
]

dim_date.head()

,date_id,arrival_date,year,month,month_name,day,quarter,is_weekend
0,20150701,2015-07-01,2015,7,July,1,3,False
1,20150702,2015-07-02,2015,7,July,2,3,False
2,20150703,2015-07-03,2015,7,July,3,3,False
3,20150704,2015-07-04,2015,7,July,4,3,True
4,20150705,2015-07-05,2015,7,July,5,3,True


In [7]:
dim_date.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date_id       793 non-null    int32         
 1   arrival_date  793 non-null    datetime64[ns]
 2   year          793 non-null    int64         
 3   month         793 non-null    int64         
 4   month_name    793 non-null    object        
 5   day           793 non-null    int64         
 6   quarter       793 non-null    int64         
 7   is_weekend    793 non-null    bool          
dtypes: bool(1), datetime64[ns](1), int32(1), int64(4), object(1)
memory usage: 41.2+ KB


In [8]:
room_types = pd.concat([
    df["reserved_room_type"],
    df["assigned_room_type"]
]).drop_duplicates().reset_index(drop=True)

dim_room_type = pd.DataFrame({
    "room_type_code": room_types
})

dim_room_type["room_type_id"] = dim_room_type.index + 1
dim_room_type["room_type_name"] = "Room Type " + dim_room_type["room_type_code"].astype(str)

dim_room_type = dim_room_type[
    [
        "room_type_id",
        "room_type_code",
        "room_type_name"
    ]
]

dim_room_type.head()

,room_type_id,room_type_code,room_type_name
0,1,A,Room Type A
1,2,C,Room Type C
2,3,D,Room Type D
3,4,E,Room Type E
4,5,G,Room Type G


In [9]:
dim_country = (
    df[["country"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_country["country_id"] = dim_country.index + 1
dim_country = dim_country[["country_id", "country"]]

dim_country.head()


,country_id,country
0,1,GBR
1,2,PRT
2,3,USA
3,4,ESP
4,5,IRL


In [10]:
dim_market_segment = (
    df[["market_segment"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_market_segment["market_segment_id"] = dim_market_segment.index + 1

dim_market_segment = dim_market_segment[
    [
        "market_segment_id",
        "market_segment"
    ]
]

dim_market_segment.head()

,market_segment_id,market_segment
0,1,Direct
1,2,Corporate
2,3,Online TA
3,4,Offline TA/TO
4,5,Complementary


In [11]:
dim_customer_segment = (
    df[["customer_type"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_customer_segment["customer_segment_id"] = dim_customer_segment.index + 1

dim_customer_segment = dim_customer_segment[
    [
        "customer_segment_id",
        "customer_type"
    ]
]

dim_customer_segment.head()

,customer_segment_id,customer_type
0,1,Transient
1,2,Contract
2,3,Transient-Party
3,4,Group


In [12]:
dim_meal = (
    df[["meal", "meal_plan"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_meal["meal_id"] = dim_meal.index + 1

dim_meal = dim_meal[
    [
        "meal_id",
        "meal",
        "meal_plan"
    ]
]

dim_meal.head()

,meal_id,meal,meal_plan
0,1,BB,Bed and Breakfast
1,2,FB,Full Board
2,3,HB,Half Board
3,4,SC,Self Catering
4,5,Undefined,Undefined / No Meal Package


In [18]:
fact_bookings = df.copy()
fact_bookings = fact_bookings.reset_index(drop=True)

# Add hotel_id
fact_bookings = fact_bookings.merge(dim_hotel, on="hotel", how="left")

# Add date_id
fact_bookings = fact_bookings.merge(
    dim_date[["date_id", "arrival_date"]],
    on="arrival_date",
    how="left"
)

# Add country_id
fact_bookings = fact_bookings.merge(dim_country, on="country", how="left")

# Add market_segment_id
fact_bookings = fact_bookings.merge(dim_market_segment, on="market_segment", how="left")

# Add customer_segment_id
fact_bookings = fact_bookings.merge(dim_customer_segment, on="customer_type", how="left")

# Add meal_id
fact_bookings = fact_bookings.merge(dim_meal, on=["meal", "meal_plan"], how="left")

In [19]:
room_type_lookup = dim_room_type[["room_type_id", "room_type_code"]]

fact_bookings = fact_bookings.merge(
    room_type_lookup.rename(columns={
        "room_type_id": "reserved_room_type_id",
        "room_type_code": "reserved_room_type"
    }),
    on="reserved_room_type",
    how="left"
)

fact_bookings = fact_bookings.merge(
    room_type_lookup.rename(columns={
        "room_type_id": "assigned_room_type_id",
        "room_type_code": "assigned_room_type"
    }),
    on="assigned_room_type",
    how="left"
)

In [20]:
fact_bookings["booking_value"] = fact_bookings["adr"] * fact_bookings["total_nights"]

fact_bookings["estimated_revenue"] = np.where(
    fact_bookings["is_canceled"] == 0,
    fact_bookings["booking_value"],
    0
)

In [22]:
fact_bookings = fact_bookings[
    [
        "hotel_id",
        "date_id",
        "reserved_room_type_id",
        "assigned_room_type_id",
        "country_id",
        "market_segment_id",
        "customer_segment_id",
        "meal_id",
        "is_canceled",
        "booking_status",
        "lead_time",
        "total_nights",
        "total_guests",
        "adr",
        "booking_value",
        "estimated_revenue",
        "booking_changes",
        "days_in_waiting_list",
        "required_car_parking_spaces",
        "total_of_special_requests"
    ]
].reset_index(drop=True)

fact_bookings["booking_id"] = fact_bookings.index + 1

fact_bookings = fact_bookings[
    ["booking_id"] + [col for col in fact_bookings.columns if col != "booking_id"]
]

fact_bookings.head()

,booking_id,hotel_id,date_id,reserved_room_type_id,assigned_room_type_id,country_id,market_segment_id,customer_segment_id,meal_id,is_canceled,...,lead_time,total_nights,total_guests,adr,booking_value,estimated_revenue,booking_changes,days_in_waiting_list,required_car_parking_spaces,total_of_special_requests
0,1,1,20150701,1,2,1,1,1,1,0,...,7,1,1.0,75.0,75.0,75.0,0,0,0,0
1,2,1,20150701,1,1,1,2,1,1,0,...,13,1,1.0,75.0,75.0,75.0,0,0,0,0
2,3,1,20150701,1,1,1,3,1,1,0,...,14,2,2.0,98.0,196.0,196.0,0,0,0,1
3,4,1,20150701,2,2,2,1,1,1,0,...,0,2,2.0,107.0,214.0,214.0,0,0,0,0
4,5,1,20150701,2,2,2,1,1,2,0,...,9,2,2.0,103.0,206.0,206.0,0,0,0,1


In [24]:
fact_bookings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86638 entries, 0 to 86637
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   booking_id                   86638 non-null  int64  
 1   hotel_id                     86638 non-null  int64  
 2   date_id                      86638 non-null  int32  
 3   reserved_room_type_id        86638 non-null  int64  
 4   assigned_room_type_id        86638 non-null  int64  
 5   country_id                   86638 non-null  int64  
 6   market_segment_id            86638 non-null  int64  
 7   customer_segment_id          86638 non-null  int64  
 8   meal_id                      86638 non-null  int64  
 9   is_canceled                  86638 non-null  int64  
 10  booking_status               86638 non-null  object 
 11  lead_time                    86638 non-null  int64  
 12  total_nights                 86638 non-null  int64  
 13  total_guests    

In [23]:
dim_hotel.to_csv(GOLD_DIR / "dim_hotel.csv", index=False)
dim_date.to_csv(GOLD_DIR / "dim_date.csv", index=False)
dim_room_type.to_csv(GOLD_DIR / "dim_room_type.csv", index=False)
dim_country.to_csv(GOLD_DIR / "dim_country.csv", index=False)
dim_market_segment.to_csv(GOLD_DIR / "dim_market_segment.csv", index=False)
dim_customer_segment.to_csv(GOLD_DIR / "dim_customer_segment.csv", index=False)
dim_meal.to_csv(GOLD_DIR / "dim_meal.csv", index=False)
fact_bookings.to_csv(GOLD_DIR / "fact_bookings.csv", index=False)

print("Gold layer tables created successfully.")

Gold layer tables created successfully.


In [25]:
fk_columns = [
    "hotel_id",
    "date_id",
    "reserved_room_type_id",
    "assigned_room_type_id",
    "country_id",
    "market_segment_id",
    "customer_segment_id",
    "meal_id"
]

fact_bookings[fk_columns].isnull().sum()

hotel_id                 0
date_id                  0
reserved_room_type_id    0
assigned_room_type_id    0
country_id               0
market_segment_id        0
customer_segment_id      0
meal_id                  0
dtype: int64

In [26]:
print("Silver rows:", len(df))
print("Fact rows:", len(fact_bookings))

Silver rows: 86638
Fact rows: 86638


In [27]:
fact_bookings[
    [
        "total_nights",
        "total_guests",
        "adr",
        "booking_value",
        "estimated_revenue"
    ]
].describe()

,total_nights,total_guests,adr,booking_value,estimated_revenue
count,86638.000000,86638.000000,86638.000000,86638.000000,86638.000000
mean,3.653212,2.030668,107.245945,397.650626,265.098594
std,2.735742,0.790711,54.365064,369.256465,332.866140
min,1.000000,1.000000,0.000000,0.000000,0.000000
25%,2.000000,2.000000,72.900000,157.000000,0.000000
50%,3.000000,2.000000,99.000000,300.000000,170.000000
75%,5.000000,2.000000,134.437500,504.000000,379.800000
max,69.000000,55.000000,5400.000000,7590.000000,7590.000000


In [28]:
print("Negative ADR:", (fact_bookings["adr"] < 0).sum())
print("Zero or negative guests:", (fact_bookings["total_guests"] <= 0).sum())
print("Zero or negative nights:", (fact_bookings["total_nights"] <= 0).sum())

Negative ADR: 0
Zero or negative guests: 0
Zero or negative nights: 0
